# MedGemma 1.5 4B — Quantization, Comparison & LiteRT Conversion

**Goal:** Quantize `google/medgemma-1.5-4b-it` to 4-bit (NF4), compare it with the base FP16 model side-by-side on medical prompts, capture benchmarks, and then convert to LiteRT format for mobile deployment.

## Pipeline
1. Environment setup & authentication
2. Load base model (FP16) — generate responses
3. Load quantized model (4-bit NF4) — generate responses
4. Side-by-side comparison with benchmarks
5. Save merged quantized model
6. LiteRT conversion for mobile

**Recommended GPU:** L4 (24GB) or A100. Free T4 will work but slower.

**Before running:**
- Accept MedGemma terms at https://huggingface.co/google/medgemma-1.5-4b-it
- Have a Hugging Face token with read access ready

---
## Section 1 — Environment Setup

In [ ]:
# Install required packages
# Pinned versions for reproducibility and to avoid PIL/torchvision conflicts
!pip install -q -U "transformers>=4.50.0"
!pip install -q -U accelerate
!pip install -q -U bitsandbytes
!pip install -q -U peft==0.13.2
!pip install -q -U "pillow<11.0.0"  # Pinning pillow to avoid _Ink import error
!pip install -q -U huggingface_hub

print('Packages installed successfully. PLEASE RESTART RUNTIME NOW (Runtime > Restart session).')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 50.3 MB/s eta 0:00:00
Packages installed successfully. PLEASE RESTART RUNTIME NOW (Runtime > Restart session).


In [ ]:
# Verify GPU availability and capacity
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Enable GPU in Colab: Runtime > Change runtime type')

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f'GPU: {gpu_name}')
print(f'VRAM: {gpu_memory_gb:.1f} GB')
print(f'CUDA version: {torch.version.cuda}')
print(f'PyTorch version: {torch.__version__}')

if gpu_memory_gb < 16:
    print('\nWARNING: Less than 16GB VRAM. FP16 base model load may fail.')
    print('Consider using 8-bit base instead, or upgrade runtime.')

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB
CUDA version: 12.8
PyTorch version: 2.10.0+cu128


In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Ensure HF_TOKEN is stored in Colab Secrets (Recommended)
# Go to the '🔑' icon in the left panel, add a new secret named 'HF_TOKEN'

hf_token = userdata.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print('Logged in via Colab Secret')
else:
    raise ValueError("HF_TOKEN Colab Secret not found. Please set your Hugging Face token in Colab Secrets and run this cell again.")

Logged in via Colab Secret


---
## Section 2 — Configuration

In [ ]:
# Central configuration — edit here, not scattered across cells

MODEL_ID = 'google/medgemma-1.5-4b-it'

# Output paths
QUANTIZED_MODEL_DIR = '/content/medgemma_quantized_4bit'
LITERT_OUTPUT_DIR = '/content/medgemma_litert'

# Generation parameters — same for both models to ensure fair comparison
GEN_CONFIG = {
    'max_new_tokens': 256,
    'do_sample': False,        # Greedy — deterministic, fair comparison
    'temperature': 1.0,        # Ignored when do_sample=False
    'top_p': 1.0,
    'repetition_penalty': 1.0,
}

# Test prompts — medical domain, representative of actual product use cases
TEST_PROMPTS = [
    {
        'category': 'Medication Info',
        'prompt': 'What is the typical adult dose of paracetamol for fever, and what is the maximum daily dose?'
    },
    {
        'category': 'Symptom Triage',
        'prompt': 'A 45-year-old patient reports sudden chest pain radiating to the left arm, sweating, and shortness of breath. What is the most likely condition and what immediate action is recommended?'
    },
    {
        'category': 'Drug Interaction',
        'prompt': 'Is it safe to take ibuprofen with warfarin? Explain the risk briefly.'
    },
    {
        'category': 'Patient Education',
        'prompt': 'Explain Type 2 Diabetes to a newly diagnosed patient in simple terms. Include lifestyle recommendations.'
    },
    {
        'category': 'Lab Interpretation',
        'prompt': 'A patient has HbA1c of 8.2%. What does this indicate and what are the next steps?'
    },
]

print(f'Model: {MODEL_ID}')
print(f'Number of test prompts: {len(TEST_PROMPTS)}')

Model: google/medgemma-1.5-4b-it
Number of test prompts: 5


---
## Section 3 — Shared Utilities

Helper functions used for both base and quantized model evaluation.

In [ ]:
import time
import gc
import torch
from transformers import AutoProcessor


def format_prompt(processor, user_message):
    """Build a chat-formatted prompt for MedGemma using its processor."""
    messages = [
        {
            'role': 'system',
            'content': [{'type': 'text', 'text': 'You are a helpful medical assistant. Provide accurate, concise information and always recommend consulting a doctor for personal health decisions.'}]
        },
        {
            'role': 'user',
            'content': [{'type': 'text', 'text': user_message}]
        }
    ]
    return processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )


def generate_response(model, processor, prompt_text, gen_config):
    """Run generation and return response text + detailed timing metrics."""
    formatted = format_prompt(processor, prompt_text)
    inputs = processor(text=formatted, return_tensors='pt').to(model.device)
    input_token_count = inputs['input_ids'].shape[1]

    # Warm up CUDA to get accurate timing
    torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            **gen_config,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id
        )

    torch.cuda.synchronize()
    elapsed_seconds = time.perf_counter() - start_time

    # Decode only the newly generated tokens (strip the input prompt)
    generated_tokens = output[0][input_token_count:]

    # --- DEBUGGING PRINT START ---
    # Temporarily print raw tokens and their full decoded form for inspection
    print(f"DEBUG: generated_tokens (first 10): {generated_tokens[:10]}")
    raw_decoded_text = processor.decode(generated_tokens, skip_special_tokens=False)
    print(f"DEBUG: raw_decoded_text (first 100 chars): {raw_decoded_text[:100].replace('\n', '\\n')}")
    # --- DEBUGGING PRINT END ---

    # Set skip_special_tokens=True to remove unwanted special tokens like <pad>
    response_text = processor.decode(generated_tokens, skip_special_tokens=True)

    output_token_count = len(generated_tokens)
    tokens_per_second = output_token_count / elapsed_seconds if elapsed_seconds > 0 else 0

    return {
        'response': response_text.strip(),
        'input_tokens': input_token_count,
        'output_tokens': output_token_count,
        'elapsed_seconds': round(elapsed_seconds, 3),
        'tokens_per_second': round(tokens_per_second, 2),
    }


def get_model_memory_gb(model):
    """Calculate model memory footprint in GB."""
    param_size_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    total_gb = (param_size_bytes + buffer_size_bytes) / 1e9
    return round(total_gb, 3)


def free_memory():
    """Aggressively free GPU memory between model loads."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def print_gpu_memory_snapshot(label):
    """Print current GPU memory usage."""
    allocated_gb = torch.cuda.memory_allocated() / 1e9
    reserved_gb = torch.cuda.memory_reserved() / 1e9
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    print(f'[{label}] Allocated: {allocated_gb:.2f} GB | Reserved: {reserved_gb:.2f} GB | Peak: {peak_gb:.2f} GB')


print('Utility functions defined')

Utility functions defined


---
## Section 4 — Load Base Model (FP16) & Generate

We run the base model first, collect responses + metrics, then unload to free memory before loading the quantized version.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

print('Loading base model (8-bit quantized)...')
print_gpu_memory_snapshot('Before load')

processor = AutoProcessor.from_pretrained(MODEL_ID)

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    load_in_8bit=True, # Changed to 8-bit loading
    device_map='auto',
)
base_model.eval()

# Explicitly set pad_token for the tokenizer to its eos_token if not defined
if processor.tokenizer.pad_token is None and processor.tokenizer.eos_token is not None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# Ensure model's pad_token_id is set, using tokenizer's pad_token_id
# This will create the attribute if it doesn't exist in model.config
if processor.tokenizer.pad_token_id is not None:
    base_model.config.pad_token_id = processor.tokenizer.pad_token_id

base_memory_gb = get_model_memory_gb(base_model)
print(f'\nBase model loaded successfully')
print(f'Model memory footprint: {base_memory_gb} GB')
print_gpu_memory_snapshot('After load')

Loading base model (8-bit quantized)...
[Before load] Allocated: 0.00 GB | Reserved: 0.00 GB | Peak: 0.00 GB


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

TypeError: Gemma3ForConditionalGeneration.__init__() got an unexpected keyword argument 'load_in_8bit'

In [ ]:
# Generate responses for all test prompts using base model
print('=' * 70)
print('BASE MODEL (FP16) — Generation')
print('=' * 70)

base_results = []

for i, item in enumerate(TEST_PROMPTS, 1):
    print(f'\n[{i}/{len(TEST_PROMPTS)}] Category: {item["category"]}')
    print(f'Prompt: {item["prompt"][:100]}...' if len(item['prompt']) > 100 else f'Prompt: {item["prompt"]}')

    result = generate_response(base_model, processor, item['prompt'], GEN_CONFIG)
    result['category'] = item['category']
    result['prompt'] = item['prompt']
    base_results.append(result)

    print(f'Latency: {result["elapsed_seconds"]}s | Tokens: {result["output_tokens"]} | Speed: {result["tokens_per_second"]} tok/s')
    print(f'Response preview: {result["response"][:200]}...' if len(result['response']) > 200 else f'Response: {result["response"]}')

print('\n' + '=' * 70)
print('Base model evaluation complete')
print('=' * 70)

BASE MODEL (FP16) — Generation

[1/5] Category: Medication Info
Prompt: What is the typical adult dose of paracetamol for fever, and what is the maximum daily dose?
DEBUG: generated_tokens (first 10): tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')
DEBUG: raw_decoded_text (first 100 chars): <pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
Latency: 4.34s | Tokens: 256 | Speed: 58.99 tok/s
Response: 

[2/5] Category: Symptom Triage
Prompt: A 45-year-old patient reports sudden chest pain radiating to the left arm, sweating, and shortness o...
DEBUG: generated_tokens (first 10): tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')
DEBUG: raw_decoded_text (first 100 chars): <pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
Latency: 4.36s | Tokens: 256 | Speed: 58.72 tok/s
Response: 

[3/5] Category: Drug Interaction
Prompt: Is it safe to take ibuprofen with warfarin? Explain the

In [ ]:
# Free base model from memory before loading quantized version
del base_model
free_memory()
print_gpu_memory_snapshot('After base model unload')

[After base model unload] Allocated: 8.61 GB | Reserved: 8.62 GB | Peak: 8.61 GB


---
## Section 5 — Load Quantized Model (4-bit NF4) & Generate

### Why NF4?
**NF4 (Normal Float 4-bit)** is designed specifically for transformer weights which follow a normal distribution. Unlike uniform 4-bit quantization, NF4 allocates quantization levels non-uniformly — more precision where weights cluster. Result: 4x memory reduction with ~1-2% quality loss, vs plain INT4 which can lose 5-10%.

### Why double quantization?
The quantization constants themselves are quantized (nested). Saves additional ~0.4 bits per parameter with negligible quality impact.

In [ ]:
from transformers import BitsAndBytesConfig

# 4-bit NF4 quantization config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',              # NormalFloat 4-bit
    bnb_4bit_compute_dtype=torch.float16,   # Compute in FP16 for speed
    bnb_4bit_use_double_quant=True,         # Nested quantization
)

print('Loading quantized model (4-bit NF4)...')
print_gpu_memory_snapshot('Before load')

quantized_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map='auto',
)
quantized_model.eval()

quantized_memory_gb = get_model_memory_gb(quantized_model)
print(f'\nQuantized model loaded successfully')
print(f'Model memory footprint: {quantized_memory_gb} GB')
print_gpu_memory_snapshot('After load')

Loading quantized model (4-bit NF4)...
[Before load] Allocated: 8.61 GB | Reserved: 8.62 GB | Peak: 8.61 GB


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]


Quantized model loaded successfully
Model memory footprint: 3.171 GB
[After load] Allocated: 11.84 GB | Reserved: 11.91 GB | Peak: 11.85 GB


In [ ]:
# Generate responses for all test prompts using quantized model
print('=' * 70)
print('QUANTIZED MODEL (4-bit NF4) — Generation')
print('=' * 70)

quantized_results = []

for i, item in enumerate(TEST_PROMPTS, 1):
    print(f'\n[{i}/{len(TEST_PROMPTS)}] Category: {item["category"]}')
    print(f'Prompt: {item["prompt"][:100]}...' if len(item['prompt']) > 100 else f'Prompt: {item["prompt"]}')

    result = generate_response(quantized_model, processor, item['prompt'], GEN_CONFIG)
    result['category'] = item['category']
    result['prompt'] = item['prompt']
    quantized_results.append(result)

    print(f'Latency: {result["elapsed_seconds"]}s | Tokens: {result["output_tokens"]} | Speed: {result["tokens_per_second"]} tok/s')
    print(f'Response preview: {result["response"][:200]}...' if len(result['response']) > 200 else f'Response: {result["response"]}')

print('\n' + '=' * 70)
print('Quantized model evaluation complete')
print('=' * 70)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


QUANTIZED MODEL (4-bit NF4) — Generation

[1/5] Category: Medication Info
Prompt: What is the typical adult dose of paracetamol for fever, and what is the maximum daily dose?


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Latency: 4.507s | Tokens: 160 | Speed: 35.5 tok/s
Response preview: As a medical assistant, I can provide general information about paracetamol (acetaminophen).

The typical adult dose of paracetamol for fever is **1 to 2 grams (g)**, usually taken every **4 to 6 hour...

[2/5] Category: Symptom Triage
Prompt: A 45-year-old patient reports sudden chest pain radiating to the left arm, sweating, and shortness o...


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Latency: 4.483s | Tokens: 174 | Speed: 38.81 tok/s
Response preview: Based on the symptoms you described (sudden chest pain radiating to the left arm, sweating, and shortness of breath), the most likely condition is a **heart attack (myocardial infarction)**.

**Immedi...

[3/5] Category: Drug Interaction
Prompt: Is it safe to take ibuprofen with warfarin? Explain the risk briefly.


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Latency: 6.61s | Tokens: 256 | Speed: 38.73 tok/s
Response preview: <unused94>thought
Here's a thinking process for responding to the question "Is it safe to take ibuprofen with warfarin? Explain the risk briefly.":

1.  **Identify the core question:** The user wants ...

[4/5] Category: Patient Education
Prompt: Explain Type 2 Diabetes to a newly diagnosed patient in simple terms. Include lifestyle recommendati...


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Latency: 6.668s | Tokens: 256 | Speed: 38.39 tok/s
Response preview: <unused94>thought
Here's a thinking process that could lead to the explanation of Type 2 Diabetes for a newly diagnosed patient:

1.  **Identify the Target Audience and Goal:** The request is to expla...

[5/5] Category: Lab Interpretation
Prompt: A patient has HbA1c of 8.2%. What does this indicate and what are the next steps?
Latency: 6.654s | Tokens: 256 | Speed: 38.47 tok/s
Response preview: An HbA1c of 8.2% indicates that your blood sugar levels have been higher than usual over the past 2-3 months. This is a sign of prediabetes or potentially diabetes.

**What it indicates:**

* **Predia...

Quantized model evaluation complete


---
## Section 6 — Side-by-Side Comparison & Benchmarks

In [ ]:
import pandas as pd

# Build comparison dataframe
comparison_rows = []
for base, quant in zip(base_results, quantized_results):
    comparison_rows.append({
        'Category': base['category'],
        'Base Latency (s)': base['elapsed_seconds'],
        'Quant Latency (s)': quant['elapsed_seconds'],
        'Base Speed (tok/s)': base['tokens_per_second'],
        'Quant Speed (tok/s)': quant['tokens_per_second'],
        'Base Tokens': base['output_tokens'],
        'Quant Tokens': quant['output_tokens'],
    })

df_comparison = pd.DataFrame(comparison_rows)

print('=' * 70)
print('PER-PROMPT COMPARISON')
print('=' * 70)
print(df_comparison.to_string(index=False))

PER-PROMPT COMPARISON
          Category  Base Latency (s)  Quant Latency (s)  Base Speed (tok/s)  Quant Speed (tok/s)  Base Tokens  Quant Tokens
   Medication Info             4.478              4.507               57.16                35.50          256           160
    Symptom Triage             4.313              4.483               59.36                38.81          256           174
  Drug Interaction             4.304              6.610               59.49                38.73          256           256
 Patient Education             4.311              6.668               59.38                38.39          256           256
Lab Interpretation             4.314              6.654               59.35                38.47          256           256


In [ ]:
# Aggregate summary
print('\n' + '=' * 70)
print('OVERALL BENCHMARK SUMMARY')
print('=' * 70)

summary = {
    'Model size (GB)': {
        'Base (FP16)': base_memory_gb,
        'Quantized (NF4)': quantized_memory_gb,
        'Reduction': f'{(1 - quantized_memory_gb/base_memory_gb) * 100:.1f}%'
    },
    'Avg latency (s)': {
        'Base (FP16)': round(sum(r['elapsed_seconds'] for r in base_results) / len(base_results), 3),
        'Quantized (NF4)': round(sum(r['elapsed_seconds'] for r in quantized_results) / len(quantized_results), 3),
    },
    'Avg tokens/sec': {
        'Base (FP16)': round(sum(r['tokens_per_second'] for r in base_results) / len(base_results), 2),
        'Quantized (NF4)': round(sum(r['tokens_per_second'] for r in quantized_results) / len(quantized_results), 2),
    },
}

for metric, values in summary.items():
    print(f'\n{metric}:')
    for k, v in values.items():
        print(f'  {k}: {v}')


OVERALL BENCHMARK SUMMARY

Model size (GB):
  Base (FP16): 8.6
  Quantized (NF4): 3.171
  Reduction: 63.1%

Avg latency (s):
  Base (FP16): 4.344
  Quantized (NF4): 5.784

Avg tokens/sec:
  Base (FP16): 58.95
  Quantized (NF4): 37.98


In [ ]:
# Full response comparison — read actual quality
print('=' * 70)
print('FULL RESPONSE COMPARISON (Quality Check)')
print('=' * 70)

for i, (base, quant) in enumerate(zip(base_results, quantized_results), 1):
    print(f'\n{"-" * 70}')
    print(f'PROMPT {i}: [{base["category"]}]')
    print(f'{"-" * 70}')
    print(f'Q: {base["prompt"]}\n')
    print(f'>>> BASE (FP16):\n{base["response"]}\n')
    print(f'>>> QUANTIZED (NF4):\n{quant["response"]}\n')

FULL RESPONSE COMPARISON (Quality Check)

----------------------------------------------------------------------
PROMPT 1: [Medication Info]
----------------------------------------------------------------------
Q: What is the typical adult dose of paracetamol for fever, and what is the maximum daily dose?

>>> BASE (FP16):


>>> QUANTIZED (NF4):
As a medical assistant, I can provide general information about paracetamol (acetaminophen).

The typical adult dose of paracetamol for fever is **1 to 2 grams (g)**, usually taken every **4 to 6 hours** as needed.

The maximum daily dose for adults is **4 grams (g)** per 24 hours.

**It is crucial to follow the instructions on the packaging and always consult your doctor or pharmacist before taking paracetamol, especially if you have any underlying health conditions or are taking other medications.**

**Never exceed the maximum daily dose.** Taking too much paracetamol can cause serious liver damage.

Remember, this information is for general

In [ ]:
# Save benchmark results to disk for later reference
import json

benchmark_report = {
    'model_id': MODEL_ID,
    'gpu': gpu_name,
    'generation_config': GEN_CONFIG,
    'summary': summary,
    'base_results': base_results,
    'quantized_results': quantized_results,
}

with open('/content/benchmark_report.json', 'w') as f:
    json.dump(benchmark_report, f, indent=2)

print('Benchmark report saved to /content/benchmark_report.json')

Benchmark report saved to /content/benchmark_report.json


---
## Section 7 — Save Quantized Model to Disk

We save the quantized weights so we can re-use them for LiteRT conversion without re-quantizing.

In [ ]:
import os

os.makedirs(QUANTIZED_MODEL_DIR, exist_ok=True)

print(f'Saving quantized model to: {QUANTIZED_MODEL_DIR}')
quantized_model.save_pretrained(QUANTIZED_MODEL_DIR)
processor.save_pretrained(QUANTIZED_MODEL_DIR)

# Verify files
saved_files = os.listdir(QUANTIZED_MODEL_DIR)
total_size_mb = sum(
    os.path.getsize(os.path.join(QUANTIZED_MODEL_DIR, f))
    for f in saved_files
) / 1e6

print(f'\nSaved {len(saved_files)} files, total size: {total_size_mb:.1f} MB')
for f in sorted(saved_files):
    size_mb = os.path.getsize(os.path.join(QUANTIZED_MODEL_DIR, f)) / 1e6
    print(f'  {f}: {size_mb:.1f} MB')

Saving quantized model to: /content/medgemma_quantized_4bit


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved 7 files, total size: 3262.3 MB
  chat_template.jinja: 0.0 MB
  config.json: 0.0 MB
  generation_config.json: 0.0 MB
  model.safetensors: 3229.0 MB
  processor_config.json: 0.0 MB
  tokenizer.json: 33.4 MB
  tokenizer_config.json: 0.0 MB


In [ ]:
# Free quantized model from memory before LiteRT conversion
del quantized_model
free_memory()
print_gpu_memory_snapshot('After quantized model unload')

[After quantized model unload] Allocated: 8.61 GB | Reserved: 8.62 GB | Peak: 8.61 GB


---
# Part 2 — Mobile Deployment via GGUF

**Why GGUF instead of LiteRT?**

LiteRT conversion for MedGemma 1.5 multimodal (text + image) is not yet stable. Google is working on it but no official release exists.

**GGUF** is the battle-tested alternative:
- Works with `llama.cpp` runtime (mobile bindings exist for Android + iOS)
- Supports both text AND image inputs
- Unsloth team has already converted MedGemma 1.5 to GGUF — we just download it
- Multiple quantization levels available (Q4, Q5, Q8)

We will download, verify, and smoke-test the GGUF so we know it works end-to-end before handing to the app team.

## Section 7 — Download Pre-Quantized GGUF from Unsloth

In [ ]:
from huggingface_hub import hf_hub_download
import os

GGUF_OUTPUT_DIR = '/content/medgemma_gguf'
os.makedirs(GGUF_OUTPUT_DIR, exist_ok=True)

# Q4_K_M is the recommended balance of size and quality for mobile
# Alternatives: Q5_K_M (better quality, ~3GB), Q8_0 (best quality, ~4.5GB)
GGUF_FILENAME = 'medgemma-1.5-4b-it-Q4_K_M.gguf'

print(f'Downloading {GGUF_FILENAME}...')
print('This is a ~2.5 GB file, will take a few minutes.\n')

gguf_path = hf_hub_download(
    repo_id='unsloth/medgemma-1.5-4b-it-GGUF',
    filename=GGUF_FILENAME,
    local_dir=GGUF_OUTPUT_DIR,
)

file_size_gb = os.path.getsize(gguf_path) / 1e9
print(f'\nDownload complete')
print(f'Path: {gguf_path}')
print(f'Size: {file_size_gb:.2f} GB')

This is a ~2.5 GB file, will take a few minutes.



medgemma-1.5-4b-it-Q4_K_M.gguf:   0%|          | 0.00/2.49G [00:00<?, ?B/s]


Download complete
Path: /content/medgemma_gguf/medgemma-1.5-4b-it-Q4_K_M.gguf
Size: 2.49 GB


## Section 8 — Smoke Test the GGUF with llama.cpp

Verify the GGUF model actually runs and produces sensible medical responses.

We use `llama-cpp-python` — the Python binding for llama.cpp. Same runtime that will run on mobile, just with Python interface for testing.

In [ ]:
# Install llama-cpp-python with CUDA support for faster testing on GPU
!CMAKE_ARGS='-DGGML_CUDA=on' pip install -q llama-cpp-python --upgrade --force-reinstall --no-cache-dir

print('llama-cpp-python installed with CUDA support')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 MB 380.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 604.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 832.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 352.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 621.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have num

In [ ]:
from llama_cpp import Llama

# Load GGUF model — same way the mobile app will load it
print('Loading GGUF model...')

llm = Llama(
    model_path=gguf_path,
    n_ctx=2048,              # Context window size
    n_gpu_layers=-1,         # -1 = offload all to GPU (remove on mobile)
    n_threads=4,             # CPU threads (mobile will use 2-4)
    verbose=False,
)

print('GGUF model loaded successfully')

Loading GGUF model...


llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


GGUF model loaded successfully


In [ ]:
# Test GGUF on the same medical prompts we used earlier
# This tells us if quality is preserved after quantization

print('=' * 70)
print('GGUF MODEL — Smoke Test on Medical Prompts')
print('=' * 70)

gguf_results = []

for i, item in enumerate(TEST_PROMPTS, 1):
    print(f'\n[{i}/{len(TEST_PROMPTS)}] Category: {item["category"]}')
    print(f'Prompt: {item["prompt"][:100]}...' if len(item['prompt']) > 100 else f'Prompt: {item["prompt"]}')

    # Format using MedGemma chat template
    system_msg = 'You are a helpful medical assistant. Provide accurate, concise information and always recommend consulting a doctor for personal health decisions.'
    formatted_prompt = f'<start_of_turn>user\n{system_msg}\n\n{item["prompt"]}<end_of_turn>\n<start_of_turn>model\n'

    start_time = time.perf_counter()
    output = llm(
        formatted_prompt,
        max_tokens=256,
        temperature=0.0,
        stop=['<end_of_turn>'],
    )
    elapsed = time.perf_counter() - start_time

    response_text = output['choices'][0]['text'].strip()
    tokens_generated = output['usage']['completion_tokens']
    tokens_per_sec = tokens_generated / elapsed if elapsed > 0 else 0

    result = {
        'category': item['category'],
        'prompt': item['prompt'],
        'response': response_text,
        'elapsed_seconds': round(elapsed, 3),
        'output_tokens': tokens_generated,
        'tokens_per_second': round(tokens_per_sec, 2),
    }
    gguf_results.append(result)

    print(f'Latency: {result["elapsed_seconds"]}s | Tokens: {result["output_tokens"]} | Speed: {result["tokens_per_second"]} tok/s')
    print(f'Response preview: {response_text[:200]}...' if len(response_text) > 200 else f'Response: {response_text}')

print('\n' + '=' * 70)
print('GGUF smoke test complete')
print('=' * 70)

GGUF MODEL — Smoke Test on Medical Prompts

[1/5] Category: Medication Info
Prompt: What is the typical adult dose of paracetamol for fever, and what is the maximum daily dose?
Latency: 1.182s | Tokens: 256 | Speed: 216.53 tok/s
Response preview: As a medical assistant, I can provide general information, but it's crucial to remember that I cannot give medical advice. **Always consult your doctor or pharmacist for personalized recommendations a...

[2/5] Category: Symptom Triage
Prompt: A 45-year-old patient reports sudden chest pain radiating to the left arm, sweating, and shortness o...
Latency: 0.959s | Tokens: 216 | Speed: 225.15 tok/s
Response preview: As a medical assistant, I cannot provide a diagnosis. However, based on the symptoms you described (sudden chest pain radiating to the left arm, sweating, and shortness of breath), this is a **medical...

[3/5] Category: Drug Interaction
Prompt: Is it safe to take ibuprofen with warfarin? Explain the risk briefly.
Latency: 1.134s | T

## Section 9 — Three-Way Comparison (Base vs NF4 vs GGUF)

Now we can compare all three versions to see where quality drops are acceptable for mobile deployment.

In [ ]:
# Three-way comparison table
three_way_rows = []
for base, quant, gguf in zip(base_results, quantized_results, gguf_results):
    three_way_rows.append({
        'Category': base['category'],
        'Base (s)': base['elapsed_seconds'],
        'NF4 (s)': quant['elapsed_seconds'],
        'GGUF Q4 (s)': gguf['elapsed_seconds'],
        'Base (tok/s)': base['tokens_per_second'],
        'NF4 (tok/s)': quant['tokens_per_second'],
        'GGUF (tok/s)': gguf['tokens_per_second'],
    })

df_three = pd.DataFrame(three_way_rows)

print('=' * 80)
print('THREE-WAY PERFORMANCE COMPARISON')
print('=' * 80)
print(df_three.to_string(index=False))

THREE-WAY PERFORMANCE COMPARISON
          Category  Base (s)  NF4 (s)  GGUF Q4 (s)  Base (tok/s)  NF4 (tok/s)  GGUF (tok/s)
   Medication Info     4.478    4.507        1.182         57.16        35.50        216.53
    Symptom Triage     4.313    4.483        0.959         59.36        38.81        225.15
  Drug Interaction     4.304    6.610        1.134         59.49        38.73        225.72
 Patient Education     4.311    6.668        1.132         59.38        38.39        226.11
Lab Interpretation     4.314    6.654        1.134         59.35        38.47        225.84


In [ ]:
# Quality comparison — read actual responses side by side
print('=' * 80)
print('QUALITY COMPARISON — All Three Versions')
print('=' * 80)

for i, (base, quant, gguf) in enumerate(zip(base_results, quantized_results, gguf_results), 1):
    print(f'\n{"-" * 80}')
    print(f'PROMPT {i}: [{base["category"]}]')
    print(f'{"-" * 80}')
    print(f'Q: {base["prompt"]}\n')
    print(f'>>> BASE (FP16):\n{base["response"]}\n')
    print(f'>>> QUANTIZED (NF4):\n{quant["response"]}\n')
    print(f'>>> GGUF (Q4_K_M) — What mobile will use:\n{gguf["response"]}\n')

QUALITY COMPARISON — All Three Versions

--------------------------------------------------------------------------------
PROMPT 1: [Medication Info]
--------------------------------------------------------------------------------
Q: What is the typical adult dose of paracetamol for fever, and what is the maximum daily dose?

>>> BASE (FP16):


>>> QUANTIZED (NF4):
As a medical assistant, I can provide general information about paracetamol (acetaminophen).

The typical adult dose of paracetamol for fever is **1 to 2 grams (g)**, usually taken every **4 to 6 hours** as needed.

The maximum daily dose for adults is **4 grams (g)** per 24 hours.

**It is crucial to follow the instructions on the packaging and always consult your doctor or pharmacist before taking paracetamol, especially if you have any underlying health conditions or are taking other medications.**

**Never exceed the maximum daily dose.** Taking too much paracetamol can cause serious liver damage.

Remember, this informa

In [ ]:
# Google Drive mount karo
from google.colab import drive
drive.mount('/content/drive')

# Folder banao Drive mein
import os
os.makedirs('/content/drive/MyDrive/MedGemma_Models', exist_ok=True)

# GGUF file copy karo
import shutil
shutil.copy(
    '/content/medgemma_gguf/medgemma-1.5-4b-it-Q4_K_M.gguf',
    '/content/drive/MyDrive/MedGemma_Models/'
)

print('Done! File saved to Google Drive')

Final benchmark report saved to /content/final_benchmark_report.json


---
## Final Deliverables

| Artifact | Location | Purpose |
|----------|----------|----------|
| Benchmark report | `/content/final_benchmark_report.json` | Quality + speed across all 3 versions |
| Quantized model (NF4) | `/content/medgemma_quantized_4bit/` | GPU/laptop inference |
| GGUF model | `/content/medgemma_gguf/medgemma-1.5-4b-it-Q4_K_M.gguf` | **Mobile deployment** |

## Next Steps for App Team

1. **Give the app team:** The GGUF file (~2.5 GB)
2. **They integrate llama.cpp** on mobile:
   - Android: `llama.cpp` JNI bindings
   - iOS: `llama.cpp` Swift wrapper
3. **For image inputs** (X-ray, prescription, skin):
   - Use MedGemma's multimodal chat template
   - Pass image path + text prompt to llama.cpp
   - Expect 5-15 seconds per image on mid-range phones
4. **First run:** Model downloads on first app launch (~2.5 GB)
5. **After that:** All inference is 100% offline

## Prompt Template for App Team

```
<start_of_turn>user
{system_message}

{user_query}<end_of_turn>
<start_of_turn>model
```

Stop token: `<end_of_turn>`